In [1]:
import cv2
import shutil
import numpy as np
import os
from ultralytics import YOLO
import time

In [2]:
# source paths

rgb_source_path = "/home/hassaan/Downloads/train_2500/images_final/"
dpt_source_path = "/home/hassaan/Downloads/train_2500/dpt_final/"

# # trial 2
rgb_destination_path = "/home/hassaan/Downloads/try_again_2/rgb_split_2.1/"
dpt_destination_path = "/home/hassaan/Downloads/try_again_2/dataset_2.1/images/"
# text_destination_path = "/home/hassaan/Downloads/try_again_2/dataset_2.1/labels/"


rgb_files_list = os.listdir(rgb_source_path)
# rgb_files_list.sort(key=lambda x: int(x.split(".")[0]))

In [3]:
pose_model = YOLO("models/yolov8m-pose.pt")
object_model = YOLO("models/yolov8m.pt")
segment_model = YOLO("models/yolov8m-seg.pt")

In [4]:
# utils imported
from utils.box_splitting import split_people_bboxes
from utils.object_detection import object_detection
# from utils.bbox_area import bbox_area
# from utils.check_overlap import check_overlap_for_2

In [ ]:
index = 0
goal = len(rgb_files_list)

In [13]:
for file in rgb_files_list[500:]:
    rgb_path = rgb_source_path + file  # /home/hassaan/Downloads/allFinal/0.png
    depth_path = dpt_source_path + file  # /home/hassaan/Downloads/DPT2/0.png
    print(rgb_path, ": source")
    # read image
    rgb_frame = cv2.imread(rgb_path)
    depth_frame = cv2.imread(depth_path)

    rgb_frame, bboxes, confs = object_detection(
        frame=rgb_frame, depth_frame=depth_frame, model=segment_model, pose_model = pose_model
    )
    print(bboxes)
    # print("-" * 30, "temp")
    # for i in bboxes:
    #     index += 1
    #     print(index)

    depth_sections, rgb_sections = split_people_bboxes(
        depth_frame=depth_frame, rgb_frame=rgb_frame, bboxes=bboxes, confs=confs
    )

    if len(depth_sections) == 0:
        print("ALL PERSON BBOXES ARE TINY")

    elif len(depth_sections) == len(rgb_sections):
        # for each section
        for n in range(len(depth_sections)):

            # write depth section to dataset/images/
            try:
                cv2.imwrite(
                    f"{dpt_destination_path}{index}.png", np.array(depth_sections[n])
                )
                cv2.imwrite(
                    f"{rgb_destination_path}{index}.png", np.array(rgb_sections[n])
                )
                print(f"{rgb_destination_path}{index}.png")
            except Exception as err:
                print(type(depth_sections[n]))
                print(rgb_path, depth_path)
                print("#8#8#", err, "#8#8#")
                continue

            # write rgb section to split_rgb/
            index += 1
    print("index:", index)
    print("=" * 30)
    if index % 30 == 0 :
        print("sleeping...")
        time.sleep(3)

/home/hassaan/Downloads/train_2500/images_final/500.png : source

0: 384x640 1 person, 1156.0ms
Speed: 3.2ms preprocess, 1156.0ms inference, 4.1ms postprocess per image at shape (1, 3, 384, 640)
[[275, 39, 617, 652]]
CASE 1
/home/hassaan/Downloads/try_again_2/rgb_split_2.1/577.png
index: 578
/home/hassaan/Downloads/train_2500/images_final/501.png : source

0: 384x640 3 persons, 2 cars, 1 truck, 2 horses, 1372.6ms
Speed: 4.4ms preprocess, 1372.6ms inference, 27.3ms postprocess per image at shape (1, 3, 384, 640)
[]
CASE 5
else case in box splitting, number of bboxes: 0
ALL PERSON BBOXES ARE TINY
index: 578
/home/hassaan/Downloads/train_2500/images_final/502.png : source

0: 384x640 2 persons, 2 cars, 1 skateboard, 1231.1ms
Speed: 2.8ms preprocess, 1231.1ms inference, 13.4ms postprocess per image at shape (1, 3, 384, 640)
[[265, 83, 1010, 616]]
CASE 1
/home/hassaan/Downloads/try_again_2/rgb_split_2.1/578.png
index: 579
/home/hassaan/Downloads/train_2500/images_final/503.png : source

0: 

In [ ]:
# # TEST IMAGE FOR PROCESS.
# file = "57.png"
# index = 0

# rgb_path = rgb_source_path + file  # /home/hassaan/Downloads/allFinal/0.png
# depth_path = dpt_source_path + file  # /home/hassaan/Downloads/DPT2/0.png

# # read image
# rgb_frame = cv2.imread(rgb_path)
# depth_frame = cv2.imread(depth_path)

# bboxes, confs = object_detection(rgb_frame, object_model)

# # depth_sections, rgb_sections = split_people_bboxes(
# #     depth_frame=depth_frame, rgb_frame=rgb_frame, bboxes=bboxes, confs=confs
# # )

# # # try:
# # #     split_people_bboxes(
# # #         depth_frame=depth_frame, rgb_frame=rgb_frame, bboxes=bboxes, confs=confs
# # #     )
# # # except Exception as err:
# # #     print(err)


# # if len(depth_sections) == len(rgb_sections):

# #     # for each section
# #     for n in range(len(depth_sections)):

# #         # write depth section to dataset/images/
# #         cv2.imwrite(
# #             f"/home/hassaan/Downloads/dataset2/images/{index}.png",
# #             np.array(depth_sections[n]),
# #         )

# #         # write rgb section to split_rgb/
# #         cv2.imwrite(
# #             f"/home/hassaan/Downloads/rgb_split2/{index}.png",
# #             np.array(rgb_sections[n]),
# #         )

# #         print(f"{rgb_destination_path}{index}.png")
# #         index += 1
# # print(index)

In [7]:
# # test
# print(bboxes)
# # check_overlap_for_2(bboxes[0], bboxes[1])
# # bbox_area(bboxes[0]), bbox_area(bboxes[1])
# # check_overlap_for_2([996, 80, 1234, 1069], [1140, 94, 1474, 1068])
# bbox1, bbox2, bbox3, bbox4 = bboxes
# area12 = check_overlap_for_2(bbox1, bbox2)
# area13 = check_overlap_for_2(bbox1, bbox3)
# area14 = check_overlap_for_2(bbox1, bbox4)

# area23 = check_overlap_for_2(bbox2, bbox3)
# area24 = check_overlap_for_2(bbox2, bbox4)

# area34 = check_overlap_for_2(bbox3, bbox4)
# print(area12, area13, area14, area23, area24, area34)

[[316, 145, 640, 1068], [601, 120, 886, 1068], [996, 80, 1234, 1069], [1140, 94, 1474, 1068]]
35997 -328588 -461500 -104280 -240792 91556


In [5]:
rgb_path = "/home/hassaan/Downloads/train_2500/images_final/1265.png"
dpt_path = "/home/hassaan/Downloads/train_2500/images_final/1265.png"
image = cv2.imread(rgb_path)
depth = cv2.imread(dpt_path)

rgb_frame, bboxes, confs = object_detection(image,depth,segment_model,pose_model)




0: 384x640 4 persons, 2 sheeps, 1636.5ms
Speed: 4.4ms preprocess, 1636.5ms inference, 14.8ms postprocess per image at shape (1, 3, 384, 640)


In [ ]:
# for i in range(len(bboxes)):
#     # xCenter, yCenter, height, width = bboxes[i]
#     # x1 = int(xCenter - width / 2)
#     # y1 = int(yCenter - height / 2)
#     print(bboxes[i])
#     x1, y1, x2, y2 = bboxes[i]
#     cropedImage = image[y1:y2, x1:x2]
#     while True:
#         cv2.imshow(f"{i} image: ", cropedImage)
        
#         if cv2.waitKey(10) & 0xFF == ord("q"):
#             break
#     cv2.destroyAllWindows()

In [ ]:
# while True:
#     cv2.imshow(f"{i} image: ", rgb_frame)
#     if cv2.waitKey(10) & 0xFF == ord("q"):
#         break